# MTG-S (Meteosat Third Generation Sounder) / IRS (InfraRed Sounder)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=mtg/mtg-irs). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1

Level 1: calibrated, geolocated infrared spectra (uncompressed and principal-component compressed), split by long-wave and mid-wave bands.

Looks for IRS collections in the EUMETSAT Data Store (title search) and downloads the first product of the window. Needs a free account and consumer key/secret, asked at run time. IRS products are large.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collections = [c for c in datastore.collections if "irs level" in str(c.title).lower()]
print([(c.collection_id, str(c.title)) for c in collections])
if not collections:
    raise SystemExit("Collection not found in the Data Store yet - check data.eumetsat.int")
collection = collections[0]

products = list(collection.search(dtstart=dt.datetime(2026, 10, 1, 12, 0), dtend=dt.datetime(2026, 10, 1, 12, 30)))
print(len(products), "IRS products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)
print(sorted(out.rglob('*.nc'))[:5])

## L2

Level 2: retrieved atmospheric profiles of temperature and humidity (sounding vertical profiles) and derived products.

Looks for IRS collections in the EUMETSAT Data Store (title search) and downloads the first product of the window. Needs a free account and consumer key/secret, asked at run time. IRS products are large.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collections = [c for c in datastore.collections if "irs level" in str(c.title).lower()]
print([(c.collection_id, str(c.title)) for c in collections])
if not collections:
    raise SystemExit("Collection not found in the Data Store yet - check data.eumetsat.int")
collection = collections[0]

products = list(collection.search(dtstart=dt.datetime(2026, 10, 1, 12, 0), dtend=dt.datetime(2026, 10, 1, 12, 30)))
print(len(products), "IRS products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)
print(sorted(out.rglob('*.nc'))[:5])